In [82]:
import sqlite3
from datetime import datetime

In [83]:
conn = sqlite3.connect('Beers.db')
cursor = conn.cursor()

In [84]:
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS CERVEZAS (
        CodC TEXT PRIMARY KEY,
        Envase TEXT,
        Capacidad REAL,
        Stock INTEGER
    );
    
    CREATE TABLE IF NOT EXISTS BARES (
        CodB TEXT PRIMARY KEY,
        Cif TEXT,
        Nombre TEXT,
        Localidad TEXT
    );
    
    CREATE TABLE IF NOT EXISTS EMPLEADOS (
        CodE INTEGER PRIMARY KEY,
        Nombre TEXT,
        Sueldo REAL
    );
    
    CREATE TABLE IF NOT EXISTS REPARTO (
        CodE INTEGER,
        CodB TEXT,
        CodC TEXT,
        Fecha TEXT,
        Cantidad INTEGER,
        PRIMARY KEY (CodE, CodB, CodC, Fecha),
        FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
        FOREIGN KEY (CodB) REFERENCES BARES(CodB),
        FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
    );
''')

In [85]:
cursor.executemany('INSERT OR REPLACE INTO CERVEZAS VALUES (?, ?, ?, ?)', [
    ('01', 'Botella', 0.2, 3600),
    ('02', 'Botella', 0.33, 1200),
    ('03', 'Lata', 0.33, 2400),
    ('04', 'Botella', 1, 288),
    ('05', 'Barril', 60, 30)
])

cursor.executemany('INSERT OR REPLACE INTO BARES VALUES (?, ?, ?, ?)', [
    ('001', '11111111X', 'Stop', 'Villa Botijo'),
    ('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
    ('003', '-', 'Club Social', 'Las Ranas'),
    ('004', '33333333Z', 'Otra Ronda', 'La Esponja')
])

cursor.executemany('INSERT OR REPLACE INTO EMPLEADOS VALUES (?, ?, ?)', [
    (1, 'Prudencio Caminero', 120000),
    (2, 'Vicente Merario', 110000),
    (3, 'Valentin Siempre', 100000)
])

cursor.executemany('INSERT OR REPLACE INTO REPARTO VALUES (?, ?, ?, ?, ?)', [
    (1, '001', '01', '2005-10-21', 240),
    (1, '001', '02', '2005-10-21', 48),
    (1, '002', '03', '2005-10-22', 60),
    (1, '004', '05', '2005-10-22', 4),
    (2, '002', '03', '2005-10-22', 48),
    (2, '002', '05', '2005-10-23', 2),
    (2, '004', '01', '2005-10-23', 480),
    (2, '004', '02', '2005-10-24', 72),
    (3, '003', '03', '2005-10-24', 48),
    (3, '003', '04', '2005-10-25', 20)
])

conn.commit()

In [86]:
def run_query(query, description):
    print(f"{description}")
    cursor.execute(query)
    rows = cursor.fetchall()
    cols = [desc[0] for desc in cursor.description]
    print(f"{' | '.join(cols)}")
    for row in rows:
        print(' | '.join(str(v) for v in row))

In [87]:
run_query(
    '''
    SELECT DISTINCT E.Nombre
    FROM EMPLEADOS E
    JOIN REPARTO R ON E.CodE = R.CodE
    JOIN BARES B ON R.CodB = B.CodB
    WHERE B.Nombre = 'Stop'
      AND R.Fecha BETWEEN '2005-10-17' AND '2005-10-23'
    ''', 
    '1. Empleados que repartieron al bar Stop (17-23 oct 2005)')

1. Empleados que repartieron al bar Stop (17-23 oct 2005)
Nombre
Prudencio Caminero


In [88]:
run_query(
    '''
    SELECT DISTINCT B.Cif, B.Nombre, B.Localidad
    FROM BARES B
    JOIN REPARTO R ON B.CodB = R.CodB
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE C.Envase = 'Botella' AND C.Capacidad < 1
    ORDER BY B.Localidad
    ''', 
    '2. Bares con reparto de Botella < 1L, ordenados por localidad')

2. Bares con reparto de Botella < 1L, ordenados por localidad
Cif | Nombre | Localidad
33333333Z | Otra Ronda | La Esponja
11111111X | Stop | Villa Botijo


In [89]:
run_query(
    '''
    SELECT B.Nombre AS Bar, C.Envase, C.Capacidad, R.Fecha, R.Cantidad
    FROM REPARTO R
    JOIN BARES B ON R.CodB = B.CodB
    JOIN CERVEZAS C ON R.CodC = C.CodC
    JOIN EMPLEADOS E ON R.CodE = E.CodE
    WHERE E.Nombre = 'Prudencio Caminero'
    ''', 
    '3. Repartos de Prudencio Caminero')

3. Repartos de Prudencio Caminero
Bar | Envase | Capacidad | Fecha | Cantidad
Stop | Botella | 0.2 | 2005-10-21 | 240
Stop | Botella | 0.33 | 2005-10-21 | 48
Las Vegas | Lata | 0.33 | 2005-10-22 | 60
Otra Ronda | Barril | 60.0 | 2005-10-22 | 4


In [90]:
run_query(
    '''
    SELECT DISTINCT B.Nombre
    FROM BARES B
    JOIN REPARTO R ON B.CodB = R.CodB
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE C.Envase = 'Botella' AND C.Capacidad IN (0.2, 0.33)
    ''', 
    '4. Bares con botella 0.2 o 0.33')

4. Bares con botella 0.2 o 0.33
Nombre
Stop
Otra Ronda


In [91]:
run_query(
    '''
    SELECT E.Nombre
    FROM EMPLEADOS E
    JOIN REPARTO R ON E.CodE = R.CodE
    JOIN BARES B ON R.CodB = B.CodB
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE C.Envase = 'Botella'
      AND B.Nombre IN ('Stop', 'Las Vegas')
    GROUP BY E.Nombre
    HAVING COUNT(DISTINCT B.Nombre) = 2
    ''', 
    '5. Empleados que repartieron a Stop y Las Vegas con botella')

5. Empleados que repartieron a Stop y Las Vegas con botella
Nombre


In [92]:
run_query(
    '''
    SELECT E.Nombre, COUNT(*) AS NumViajes
    FROM EMPLEADOS E
    JOIN REPARTO R ON E.CodE = R.CodE
    JOIN BARES B ON R.CodB = B.CodB
    WHERE B.Localidad != 'Villa Botijo'
    GROUP BY E.Nombre
    ''', 
    '6. Viajes fuera de Villa Botijo por empleado')

6. Viajes fuera de Villa Botijo por empleado
Nombre | NumViajes
Prudencio Caminero | 1
Valentin Siempre | 2
Vicente Merario | 2


In [93]:
run_query(
    '''
    SELECT B.Nombre, B.Localidad, SUM(C.Capacidad * R.Cantidad) AS TotalLitros
    FROM BARES B
    JOIN REPARTO R ON B.CodB = R.CodB
    JOIN CERVEZAS C ON R.CodC = C.CodC
    GROUP BY B.Nombre, B.Localidad
    ORDER BY TotalLitros DESC
    LIMIT 1
    ''', 
    '7. Bar que más litros ha comprado')

7. Bar que más litros ha comprado
Nombre | Localidad | TotalLitros
Otra Ronda | La Esponja | 359.76


In [94]:
run_query(
    '''
    SELECT B.Nombre
    FROM BARES B
    WHERE NOT EXISTS (
        SELECT *
        FROM CERVEZAS C
        WHERE C.Envase = 'Botella' AND C.Capacidad < 1
          AND NOT EXISTS (
              SELECT *
              FROM REPARTO R
              WHERE R.CodB = B.CodB AND R.CodC = C.CodC
          )
    )
    ''', 
    '8. Bares que han adquirido todas las botellas < 1L')

8. Bares que han adquirido todas las botellas < 1L
Nombre
Stop
Otra Ronda


In [95]:
cursor.execute(
    '''
    UPDATE EMPLEADOS
    SET Sueldo = Sueldo * 1.05
    WHERE CodE = (
        SELECT E.CodE
        FROM EMPLEADOS E
        JOIN REPARTO R ON E.CodE = R.CodE
        GROUP BY E.CodE
        ORDER BY COUNT(DISTINCT R.Fecha) DESC
        LIMIT 1
    )
    ''')
conn.commit()
print('9. Sueldo actualizado (+5%) al empleado con más días trabajados')

9. Sueldo actualizado (+5%) al empleado con más días trabajados


In [96]:
cursor.execute(
    '''
    INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
    VALUES (2, '001', '03', '2005-10-26', 48)
    ''')
conn.commit()
print('10. Nuevo reparto insertado: Vicente Merario -> Stop, 48 Latas, 2005-10-26')

10. Nuevo reparto insertado: Vicente Merario -> Stop, 48 Latas, 2005-10-26


In [98]:
conn.close()